# Day 2 目标：把 Demo 工程化

今天你不需要加复杂功能，也不需要接 LLM / RAG。
今天只做一件事：

把 Day 1 的 day1_minimal_graph.py 拆成标准项目结构，并能通过 run_cli.py 运行。

# 一、Day 2 最终目录结构

你今天做完后，项目应该长这样：

今天重点是这几个文件：

| 文件            | 作用            |
| ------------- | ------------- |
| `state.py`    | 定义 AgentState |
| `nodes.py`    | 放所有节点函数       |
| `router.py`   | 放路由函数         |
| `workflow.py` | 组装 LangGraph  |
| `run_cli.py`  | 命令行入口         |


# 二、先创建目录

在 VSCode 终端运行：

cd F:\ResearchAgent

然后创建目录：

mkdir src
mkdir src\research_agent
mkdir src\research_agent\graph
mkdir notebooks

创建初始化文件：

New-Item src\research_agent\__init__.py
New-Item src\research_agent\graph\__init__.py

如果文件已存在，报错不用管。

# 三、创建 state.py

路径：

F:\ResearchAgent\src\research_agent\graph\state.py

内容：

from typing import TypedDict


class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str

你要理解：

state.py 只负责定义整个 Agent 流程里共享的数据结构。

# 四、创建 nodes.py

路径：

F:\ResearchAgent\src\research_agent\graph\nodes.py

内容：

from .state import AgentState


def classify_task(state: AgentState) -> dict:
    """
    根据用户输入判断任务类型。
    Day 2 暂时继续使用规则分类，不接 LLM。
    """
    query = state["query"]

    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
        task_type = "experiment_analysis"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
    elif "代码" in query or "脚本" in query or "bug" in query.lower():
        task_type = "code_help"
    else:
        task_type = "general"

    return {
        "task_type": task_type
    }


def paper_node(state: AgentState) -> dict:
    return {
        "result": "这是论文问答任务，后续会接入论文 RAG。"
    }


def experiment_node(state: AgentState) -> dict:
    return {
        "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。"
    }


def dataset_node(state: AgentState) -> dict:
    return {
        "result": "这是数据集推荐任务，后续会接入数据集资料库。"
    }


def code_node(state: AgentState) -> dict:
    return {
        "result": "这是代码辅助任务，后续会接入代码解释与修改工具。"
    }


def general_node(state: AgentState) -> dict:
    return {
        "result": "这是通用科研助手任务。"
    }


def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
处理结果：{state["result"]}
""".strip()

    return {
        "final_answer": final_answer
    }

你要理解：

nodes.py 里每个函数都是 LangGraph 的一个节点。
节点接收 state，返回一个 dict 来更新 state。

# 五、创建 router.py

路径：

F:\ResearchAgent\src\research_agent\graph\router.py

内容：

from typing import Literal

from .state import AgentState


def route_task(state: AgentState) -> Literal[
    "paper_node",
    "experiment_node",
    "dataset_node",
    "code_node",
    "general_node",
]:
    """
    根据 task_type 决定下一步进入哪个节点。
    """
    task_type = state["task_type"]

    if task_type == "paper_question":
        return "paper_node"
    elif task_type == "experiment_analysis":
        return "experiment_node"
    elif task_type == "dataset_recommendation":
        return "dataset_node"
    elif task_type == "code_help":
        return "code_node"
    else:
        return "general_node"

你要理解：

router.py 不负责生成内容，只负责决定路线。

# 六、创建 workflow.py

路径：

F:\ResearchAgent\src\research_agent\graph\workflow.py

内容：

from langgraph.graph import StateGraph, START, END

from .state import AgentState
from .nodes import (
    classify_task,
    paper_node,
    experiment_node,
    dataset_node,
    code_node,
    general_node,
    final_answer_node,
)
from .router import route_task


def build_graph():
    workflow = StateGraph(AgentState)

    workflow.add_node("classify_task", classify_task)

    workflow.add_node("paper_node", paper_node)
    workflow.add_node("experiment_node", experiment_node)
    workflow.add_node("dataset_node", dataset_node)
    workflow.add_node("code_node", code_node)
    workflow.add_node("general_node", general_node)

    workflow.add_node("final_answer", final_answer_node)

    workflow.add_edge(START, "classify_task")

    workflow.add_conditional_edges(
        "classify_task",
        route_task,
        {
            "paper_node": "paper_node",
            "experiment_node": "experiment_node",
            "dataset_node": "dataset_node",
            "code_node": "code_node",
            "general_node": "general_node",
        },
    )

    workflow.add_edge("paper_node", "final_answer")
    workflow.add_edge("experiment_node", "final_answer")
    workflow.add_edge("dataset_node", "final_answer")
    workflow.add_edge("code_node", "final_answer")
    workflow.add_edge("general_node", "final_answer")

    workflow.add_edge("final_answer", END)

    return workflow.compile()

你要理解：

workflow.py 是真正组装 LangGraph 的地方。
它不写具体业务逻辑，只负责把节点和边接起来。

# 七、创建 run_cli.py

路径：

F:\ResearchAgent\run_cli.py

内容：

from src.research_agent.graph.workflow import build_graph


def run_agent(query: str) -> dict:
    graph = build_graph()

    result = graph.invoke({
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
    })

    return result


def main():
    print("ResearchAgent v0.1 CLI")
    print("输入 q / quit / exit 退出")

    while True:
        query = input("\n请输入你的科研问题：\n> ")

        if query.lower() in ["q", "quit", "exit"]:
            print("已退出。")
            break

        result = run_agent(query)

        print("\n===== Agent 输出 =====")
        print(result["final_answer"])


if __name__ == "__main__":
    main()

# 九、Day 2 加分任务：新增 report_generation

如果上面都跑通，今天做一个小扩展：

新增一个任务类型：report_generation

目标效果：

用户输入：

帮我生成组会汇报文本

输出：

任务类型：report_generation
处理结果：这是汇报生成任务，后续会接入 Report Writer。
1. 修改 state.py

不用改。

当前 State 已经够用：

class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str
2. 修改 nodes.py

在 classify_task 里新增一条规则，建议放在 代码 判断前面：

elif "汇报" in query or "PPT" in query.upper() or "组会" in query:
    task_type = "report_generation"

完整片段像这样：

if "论文" in query or "paper" in query.lower():
    task_type = "paper_question"
elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
    task_type = "experiment_analysis"
elif "数据集" in query or "dataset" in query.lower():
    task_type = "dataset_recommendation"
elif "汇报" in query or "PPT" in query.upper() or "组会" in query:
    task_type = "report_generation"
elif "代码" in query or "脚本" in query or "bug" in query.lower():
    task_type = "code_help"
else:
    task_type = "general"

然后新增节点：

def report_node(state: AgentState) -> dict:
    return {
        "result": "这是汇报生成任务，后续会接入 Report Writer。"
    }
3. 修改 router.py

把返回类型加上 "report_node"：

def route_task(state: AgentState) -> Literal[
    "paper_node",
    "experiment_node",
    "dataset_node",
    "report_node",
    "code_node",
    "general_node",
]:

然后加分支：

elif task_type == "report_generation":
    return "report_node"

完整版本：

from typing import Literal

from .state import AgentState


def route_task(state: AgentState) -> Literal[
    "paper_node",
    "experiment_node",
    "dataset_node",
    "report_node",
    "code_node",
    "general_node",
]:
    """
    根据 task_type 决定下一步进入哪个节点。
    """
    task_type = state["task_type"]

    if task_type == "paper_question":
        return "paper_node"
    elif task_type == "experiment_analysis":
        return "experiment_node"
    elif task_type == "dataset_recommendation":
        return "dataset_node"
    elif task_type == "report_generation":
        return "report_node"
    elif task_type == "code_help":
        return "code_node"
    else:
        return "general_node"
4. 修改 workflow.py

先导入 report_node：

from .nodes import (
    classify_task,
    paper_node,
    experiment_node,
    dataset_node,
    report_node,
    code_node,
    general_node,
    final_answer_node,
)

注册节点：

workflow.add_node("report_node", report_node)

条件路由映射表加一行：

"report_node": "report_node",

固定边加一行：

workflow.add_edge("report_node", "final_answer")

完整关键部分应该类似这样：

workflow.add_node("paper_node", paper_node)
workflow.add_node("experiment_node", experiment_node)
workflow.add_node("dataset_node", dataset_node)
workflow.add_node("report_node", report_node)
workflow.add_node("code_node", code_node)
workflow.add_node("general_node", general_node)
workflow.add_conditional_edges(
    "classify_task",
    route_task,
    {
        "paper_node": "paper_node",
        "experiment_node": "experiment_node",
        "dataset_node": "dataset_node",
        "report_node": "report_node",
        "code_node": "code_node",
        "general_node": "general_node",
    },
)
workflow.add_edge("paper_node", "final_answer")
workflow.add_edge("experiment_node", "final_answer")
workflow.add_edge("dataset_node", "final_answer")
workflow.add_edge("report_node", "final_answer")
workflow.add_edge("code_node", "final_answer")
workflow.add_edge("general_node", "final_answer")

#  十、Day 2 最终验收标准

今天完成这些就算过关：

1. 单文件 Demo 被拆成多个模块
2. run_cli.py 可以正常运行
3. 至少 5 种输入能进入不同任务节点
4. 你能说清楚每个文件负责什么
5. 你成功新增 report_generation 任务类型

# 十一、你今天最该理解的东西

Day 1 你理解的是：

State / Node / Edge / Conditional Edge / compile / invoke

Day 2 你要理解的是：

一个 Agent 项目应该怎么拆文件

最重要的是这张表：

| 文件            | 职责     |
| ------------- | ------ |
| `state.py`    | 定义数据结构 |
| `nodes.py`    | 定义节点行为 |
| `router.py`   | 定义路由规则 |
| `workflow.py` | 组装图    |
| `run_cli.py`  | 用户入口   |
